# Network Flow V2 — Edge-IIoTset Step 03: Split readiness and provisional 70/15/15 partitions

**Data:** `data/raw/edge_iiotset/ML-EdgeIIoT-dataset.csv` (selected ML subset; 157,800 rows, 63 columns in previous audit).

**Purpose:** verify the exact source file, check *both* target labels, investigate sequence/time context, and create reproducible **feature-group-disjoint** train/validation/test assignment. The experiment is **provisional**: identical **full 61-feature** vectors cannot cross partitions, but related packets from the same capture/attack scenario might still cross. That stronger leakage risk cannot be solved without provenance/capture identifiers. **Later feature selection can also make more rows identical, so this grouping is not sufficient for the final trained feature set.**

**Labels:** `Attack_label` (0/1) and `Attack_type` (15 total classes) are ground truth, never feature inputs. Keep labels stored separately from `X` at model training time.

This notebook does **not** normalize, delete, or change the original CSV, fit models, balance classes, or use held-out test labels for model selection. It saves a row-index split manifest and an aggregated JSON diagnostics report.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

ROOT = Path(r"D:\J26-CS-336-Network-Flow-V2\network_flow_v2")
DATA = ROOT / "data" / "raw" / "edge_iiotset" / "ML-EdgeIIoT-dataset.csv"
SPLIT_DIR = ROOT / "data" / "splits" / "edge_iiotset"
REPORT_DIR = ROOT / "reports" / "audits"
EXPECTED_SHA256 = "53101fad091af20bee815860962a5016b802ee45d456a141042f6183094c3c1a"
SEED = 42

assert DATA.is_file(), f"CSV not found: {DATA}"
print("Python:", sys.executable)
print("CSV:", DATA)
print("Size (MB):", round(DATA.stat().st_size / 1024**2, 2))

## 1. Check file integrity before assigning rows

A split manifest refers to **original CSV row positions**. If the CSV changes or rows are reordered, the manifest cannot be reused safely. Therefore we check its SHA-256 against the Step 01 audit.

In [ ]:
digest = hashlib.sha256()
with DATA.open("rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(block)
ACTUAL_SHA256 = digest.hexdigest()
print("SHA-256:", ACTUAL_SHA256)
assert ACTUAL_SHA256 == EXPECTED_SHA256, (
    "CSV differs from Step 01 audit. Do not create splits; check the file/version first."
)

df = pd.read_csv(DATA, low_memory=False)
assert df.shape == (157800, 63), f"Unexpected dataset shape: {df.shape}"
assert {"Attack_label", "Attack_type"}.issubset(df.columns)
label = pd.to_numeric(df["Attack_label"], errors="coerce")
kind = df["Attack_type"].astype("string").str.strip()
assert label.notna().all() and label.isin([0, 1]).all(), "Invalid binary label"
assert kind.notna().all() and kind.ne("").all(), "Invalid attack type"
assert (label.astype(int).eq(kind.str.casefold().ne("normal").astype(int))).all(), (
    "Binary and attack-type labels disagree"
)
feature_cols = [x for x in df.columns if x not in {"Attack_label", "Attack_type"}]
print("Validated source:", len(df), "rows;", len(feature_cols), "candidate fields;")
print("Distinct attack-type labels:", kind.nunique())

## 2. Preflight: time and ordering / capture information

The original selected ML file may be *sampled and concatenated* by attack category, and a column named `frame.time` does not guarantee reliable temporal ordering. We count label transitions and check timestamp representation. **We do not use `frame.time` as a model feature here**. No date or campaign-based split is inferred automatically.

In [ ]:
# Check how often attack classes change from one row to the next.
class_runs = int(kind.ne(kind.shift()).fillna(True).sum())
print("Label runs in CSV row order:", class_runs)
print("Class-block grouping can make row-order splitting misleading. This is a diagnostic, not proof of provenance.")

if "frame.time" in df:
    times = df["frame.time"].astype("string").str.strip()
    as_number = pd.to_numeric(times, errors="coerce")
    numeric_fraction = float(as_number.notna().mean())
    print("frame.time distinct values:", times.nunique())
    print("frame.time numeric parse percent:", round(numeric_fraction * 100, 2))
    if numeric_fraction > 0.95:
        v = as_number.dropna()
        print("Numeric range:", float(v.min()), "to", float(v.max()))
        print("Numeric units (e.g. seconds vs unix epoch) are UNVERIFIED.")
    else:
        # Display only structural hints, never network identifiers or raw payload.
        patterns = times.dropna().head(1000).str.replace(r"[0-9]", "#", regex=True)
        print("Example masked time pattern(s):")
        print(patterns.value_counts().head(3).to_string())
else:
    print("frame.time not present")

print("Capture/session identifier available? Not established from the audited 61 fields.")

## 3. Fingerprint candidate inputs and check multiclass conflicts

Hash all **non-label fields together**. Groups with the same candidate input fingerprint must never cross train/validation/test. This prevents exact-feature duplication leakage, but **does not guarantee independence between similar packets from the same session**.

Importantly, Step 02 checked binary label conflicts. This cell also checks **attack-type** conflicts, since multi-class training needs unambiguous input labels. The 64-bit fingerprint is a practical grouping aid; a hash is not a provenance or session ID.

In [ ]:
feature_hash = pd.util.hash_pandas_object(df[feature_cols], index=False).astype("uint64")
meta = pd.DataFrame({
    "fingerprint": feature_hash.to_numpy(),
    "label": label.astype("int8").to_numpy(),
    "attack_type": kind.astype(str).to_numpy()
})
agg = meta.groupby("fingerprint", sort=False).agg(
    n_rows=("label", "size"),
    binary_distinct=("label", "nunique"),
    type_distinct=("attack_type", "nunique"),
    attack_type=("attack_type", "first"),
)
conflict_binary_groups = int((agg["binary_distinct"] > 1).sum())
conflict_type_groups = int((agg["type_distinct"] > 1).sum())
print("Unique feature-fingerprint groups:", len(agg))
print("Excess observations in repeated groups:", len(meta) - len(agg))
print("Conflicting binary-label groups:", conflict_binary_groups)
print("Conflicting attack-type groups:", conflict_type_groups)
assert conflict_binary_groups == 0 and conflict_type_groups == 0, (
    "Conflicting ground truth for the same candidate inputs. Stop and inspect before splitting."
)

group_counts = agg["attack_type"].value_counts()
print("Smallest class by feature-fingerprint groups:")
display(group_counts.sort_values().head(5).rename("distinct_input_groups").to_frame())
assert group_counts.min() >= 8, "Too few independent feature groups for this stratified scheme"

## 4. Create provisional, disjoint 70% / 15% / 15% partitions

Split **unique fingerprint groups** using `Attack_type` stratification, not individual observations. After feature selection, recheck overlap using the final feature columns and ideally replace this split with a source session/capture-disjoint split. The percentages target unique groups, so the *number of rows* in each set may be slightly different from 70/15/15. All normal and each attack type should be represented in every partition.

**Why provisional?** We still cannot prove that traffic from one capture/device/campaign does not appear in multiple partitions. Do not use this as a publication-quality independent test until that concern is resolved. If capture or timeline metadata becomes available, create a new versioned, provenance-aware split instead of silently replacing this one.

In [ ]:
group_ids = agg.index.to_numpy()
group_types = agg["attack_type"].to_numpy()

train_val_groups, test_groups = train_test_split(
    group_ids,
    test_size=0.15,
    random_state=SEED,
    stratify=group_types,
)
train_val_types = agg.loc[train_val_groups, "attack_type"].to_numpy()
train_groups, val_groups = train_test_split(
    train_val_groups,
    test_size=(0.15 / 0.85),
    random_state=SEED,
    stratify=train_val_types,
)

# Explicitly verify no feature fingerprint crosses partitions.
train_set, val_set, test_set = set(train_groups), set(val_groups), set(test_groups)
assert not (train_set & val_set or train_set & test_set or val_set & test_set)
assert len(train_set | val_set | test_set) == len(agg)

split_by_group = {}
for keys, part in [(train_groups, "train"), (val_groups, "validation"), (test_groups, "test")]:
    split_by_group.update({int(key): part for key in keys})

split_names = feature_hash.map(split_by_group)
assert split_names.notna().all(), "Some observations were not assigned"

manifest = pd.DataFrame({
    "row_id": np.arange(len(df), dtype=np.int64),
    "partition": split_names.to_numpy(),
    "feature_fingerprint": feature_hash.astype(str).to_numpy(),
    "Attack_label": label.astype("int8").to_numpy(),
    "Attack_type": kind.astype(str).to_numpy(),
})

summary = manifest.groupby("partition").agg(
    rows=("row_id", "size"),
    attacks=("Attack_label", "sum"),
)
summary["normal"] = summary["rows"] - summary["attacks"]
summary["row_pct"] = (100 * summary["rows"] / len(manifest)).round(3)
summary["attack_pct"] = (100 * summary["attacks"] / summary["rows"]).round(3)
display(summary.loc[["train", "validation", "test"]])

class_split = pd.crosstab(manifest["Attack_type"], manifest["partition"])
assert {"train", "validation", "test"}.issubset(class_split.columns)
assert (class_split > 0).all().all(), "A class is missing from a partition; change split design"
display(class_split.sort_index()[["train", "validation", "test"]])

## 5. Save assignment and audit report (not the raw records)

- `data/splits/edge_iiotset/edge_iiotset_split_manifest_v1.csv` stores **row identifiers, labels, grouping fingerprints, and partitions**. This file should remain local and ignored by Git.
- `reports/audits/edge_iiotset_split_diagnostics_v1.json` stores aggregate diagnostics suitable for code review once examined for sensitive content.

There is no feature normalization and no model fitting. Later, model code must join by `row_id` after checking the same source SHA and must remove **both** label columns from `X`.

In [ ]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
manifest_path = SPLIT_DIR / "edge_iiotset_split_manifest_v1.csv"
report_path = REPORT_DIR / "edge_iiotset_split_diagnostics_v1.json"

# Avoid overwriting existing experimental artifacts. Only create if neither exists.
assert not manifest_path.exists(), f"Split manifest already exists: {manifest_path}. Review or version it first."
assert not report_path.exists(), f"Diagnostics report already exists: {report_path}. Review or version it first."

manifest.to_csv(manifest_path, index=False)
report = {
    "experiment": "Edge-IIoTset selected ML subset — provisional hash-group-disjoint split",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "source_csv": str(DATA),
    "source_sha256": ACTUAL_SHA256,
    "random_seed": SEED,
    "target_fraction_by_feature_group": {"train": 0.70, "validation": 0.15, "test": 0.15},
    "status": "PROVISIONAL — capture/session/campaign disjointness unverified",
    "reason": "Selected ML CSV has no verified capture or campaign group boundaries",
    "grouping_method": "pandas hash_pandas_object(all 61 non-label columns, index=False), uint64",
    "n_rows": int(len(df)),
    "n_distinct_feature_groups": int(len(agg)),
    "excess_duplicate_feature_rows": int(len(df)-len(agg)),
    "conflict_binary_groups": conflict_binary_groups,
    "conflict_attack_type_groups": conflict_type_groups,
    "label_runs_in_original_csv_order": class_runs,
    "row_counts": {str(k): int(v) for k,v in summary["rows"].items()},
    "attack_counts": {str(k): int(v) for k,v in summary["attacks"].items()},
    "normal_counts": {str(k): int(v) for k,v in summary["normal"].items()},
    "attack_type_by_partition": {
        str(class_name): {str(col): int(val) for col,val in row.items()}
        for class_name,row in class_split.to_dict(orient="index").items()
    },
    "warnings": [
       "This split eliminates overlap for full 61-feature vectors, not necessarily overlap after feature reduction, nor time/capture/campaign overlap.",
       "Original CSV row order and frame.time semantics are not verified as event-time split keys.",
       "Do not train on Attack_label, Attack_type, partition or feature_fingerprint.",
       "Raw payloads and high-cardinality identifiers require feature leakage and deployability review.",
       "Do not optimize models on test labels; tune with validation only.",
    ],
}
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved manifest:", manifest_path)
print("Saved diagnostics:", report_path)
print("Status: PROVISIONAL. Original dataset not modified.")

## What to share for review

After running all cells, share:

1. Label runs in original row order and `frame.time` parse-percent / masked patterns.
2. Unique feature groups and both conflict counts.
3. The three-row partition summary and per-class table.
4. Confirmation that `edge_iiotset_split_diagnostics_v1.json` was saved.

**Next:** the final model feature set may create *new* duplicate feature vectors across these provisional partitions. Re-check leakage after feature selection. Review your feature dictionary and identify a **deployment-compatible** feature subset. Build preprocessing fit **only** on training data; use validation for model selection; reserve the test labels for a later final evaluation. Prefer a new capture/session-disjoint partition if authentic capture metadata or raw PCAPs become available.